In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Set OUTPUT_ROOT to the folder where clustering results/plots should be written.
OUTPUT_ROOT = os.environ.get("OUTPUT_ROOT", "./output/kmeans")

# Function to create folder if it doesn't exist
def create_folder(folder_name):
    if not os.path.exists(folder_name):
        os.makedirs(folder_name)

# Function to perform clustering, calculate scores, and save plots
def perform_clustering_and_save(df, feature_names, folder_name):
    root = OUTPUT_ROOT + "/"
    create_folder(f"{root}{folder_name}")

    # Standardize features
    scaler = StandardScaler()
    X = scaler.fit_transform(df[feature_names])

    # Store inertia & silhouette scores
    inertia = []
    silhouette_scores = []

    for k in range(2, 11): 
        kmeans = KMeans(n_clusters=k, random_state=42, n_init="auto")
        cluster_labels = kmeans.fit_predict(X)

        # Calculate metrics
        inertia.append(kmeans.inertia_)
        silhouette = silhouette_score(X, cluster_labels)
        silhouette_scores.append(silhouette)

        # Append cluster labels to DataFrame and save
        df_clustered = df.copy()
        df_clustered['Cluster'] = cluster_labels

        create_folder(f"{root}{folder_name}/results")
        df_clustered.to_csv(f"{root}{folder_name}/results/cluster_results_k{k}.csv", index=False)

        # Compute and save mean feature values per cluster
        cluster_means = df_clustered.groupby('Cluster')[feature_names].mean().reset_index()
        
        # Round all numerical columns to 2 decimal places
        cluster_means = cluster_means.apply(lambda x: x.round(2) if x.dtype == 'float64' or x.dtype == 'int64' else x)
        
        create_folder(f"{root}{folder_name}/means")
        cluster_means.to_csv(f"{root}{folder_name}/means/cluster_means_k{k}.csv", index = False)
        
        # PCA for visualization
        pca = PCA(n_components=2)
        X_pca = pca.fit_transform(X)
        
        # Scatter plot
        create_folder(f"{root}{folder_name}/plots")
        plt.figure(figsize=(6, 5))
        scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap="tab10", edgecolors='k', alpha=0.7)
        plt.title(f"KMeans Clustering (k={k})")
        plt.xlabel("PCA1")
        plt.ylabel("PCA2")
        
        # Create categorical legend
        handles, labels = scatter.legend_elements(prop="colors", alpha=0.7)
        plt.legend(handles, labels, title="Cluster")
        
        plt.savefig(f"{root}{folder_name}/plots/kmeans_k{k}.png")
        plt.close()
        
        # Emotion plot mean
        create_folder(f"{root}{folder_name}/emotion_plots")
        
        # Apply the transformation to scale 'valence' and 'arousal' to [-1, 1]
        df_emotion = df_clustered.copy()
        # df_emotion['valence'] = df_clustered['valence'].apply(lambda x: x * 2.0 - 1.0)
        # df_emotion['arousal'] = df_clustered['arousal'].apply(lambda x: x * 2.0 - 1.0)
    
        plt.figure(figsize=(8, 6))
        # audio
        sns.scatterplot(data=df_emotion, x='valence_raw', y='arousal_raw', hue='Cluster', palette='tab10', alpha=0.7, s=80, edgecolor='black')
        # visual
        #sns.scatterplot(data=df_emotion, x='valence_mean', y='arousal_mean', hue='Cluster', palette='tab10', alpha=0.7, s=80, edgecolor='black')
        
        plt.xlabel("Valence", fontsize=12)
        plt.ylabel("Arousal", fontsize=12)
        plt.axhline(0, color='gray', linestyle='--', linewidth=0.8)  # line: arousal=0
        plt.axvline(0, color='gray', linestyle='--', linewidth=0.8)  # line: valence=0
        plt.xlim(-1, 1)  
        plt.ylim(-1, 1) 
        plt.title("Arousal-Valence Clustering")
        plt.legend(title="Cluster")
        plt.grid(True, linestyle="--", alpha=0.6)
        
        plt.savefig(f"{root}{folder_name}/emotion_plots/kmeans_k{k}.png")
        plt.close()
        
        # Emotion plot std
        create_folder(f"{root}{folder_name}/emotion_std_plots")
        
        plt.figure(figsize=(8, 6))
        # audio
        sns.scatterplot(data=df_clustered, x='std_valence', y='std_arousal', hue='Cluster', palette='tab10', s=80, edgecolor='black')
        # visual
        #sns.scatterplot(data=df_clustered, x='valence_std', y='arousal_std', hue='Cluster', palette='tab10', s=80, edgecolor='black')
        
        plt.xlabel("Valence")
        plt.ylabel("Arousal")
        plt.title("Arousal-Valence Std Clustering")
        plt.legend(title="Cluster")
        plt.grid(True, linestyle="--", alpha=0.6)
        
        plt.savefig(f"{root}{folder_name}/emotion_std_plots/kmeans_k{k}.png")
        plt.close()

    # Save elbow method plot
    plt.figure(figsize=(6, 5))
    plt.plot(range(2, 11), inertia, marker='o', linestyle='--')
    plt.xlabel("Number of Clusters (k)")
    plt.ylabel("Inertia (Elbow Score)")
    plt.title("Elbow Method")
    plt.savefig(f"{root}{folder_name}/elbow.png")
    plt.close()

    # Save silhouette scores plot
    plt.figure(figsize=(6, 5))
    plt.plot(range(2, 11), silhouette_scores, marker='s', linestyle='-')
    plt.xlabel("Number of Clusters (k)")
    plt.ylabel("Silhouette Score")
    plt.title("Silhouette Scores")
    plt.savefig(f"{root}{folder_name}/silhouette.png")
    plt.close()

In [2]:
# Base
features = ["mean_f0", "std_f0", "VUV", "speaking_rate_w", "MFCC0"]

# Base + Intonation
features1 = ["mean_f0", "std_f0", "VUV", "speaking_rate_w", "MFCC0",
            "falling_ratio", "rising_ratio", "rising-falling_ratio", "falling-rising_ratio"]

# Base + formants
features2 = ["mean_f0", "std_f0", "VUV", "speaking_rate_w", "MFCC0",
            "F1", "F2"]

# Base + formants + Intonation
features3 = ["mean_f0", "std_f0", "VUV", "speaking_rate_w", "MFCC0",
            "F1", "F2",
            "falling_ratio", "rising_ratio", "rising-falling_ratio", "falling-rising_ratio"]

# Base + AV
features4 = ["mean_f0", "std_f0", "VUV", "speaking_rate_w", "MFCC0",
            "arousal", "valence", "std_arousal", "std_valence"]

# Base + formants + Intonation + VA
features5 = ["mean_f0", "std_f0", "VUV", "speaking_rate_w", "MFCC0",
            "F1", "F2",
            "falling_ratio", "rising_ratio", "rising-falling_ratio", "falling-rising_ratio",
            "arousal", "valence", "std_arousal", "std_valence"]

In [ ]:
# A
df = pd.read_csv(os.environ.get("FEATURES_VIDEO_CSV", "./data/features_video.csv"))

# N = 398
df['date'] = pd.to_datetime(df['date'])
df1 = df[df['date'] >= pd.to_datetime('2024/09/01')]
df2 = df1[df1['date'] <= pd.to_datetime('2024/11/04')]

# Perform clustering and save results
# perform_clustering_and_save(df, features, "A")
# perform_clustering_and_save(df, features1, "A1")
# perform_clustering_and_save(df, features2, "A2")
# perform_clustering_and_save(df, features3, "A3")
# perform_clustering_and_save(df, features4, "A4")
perform_clustering_and_save(df2, features5, "audio")

# B
# df = pd.read_csv(os.environ.get("FEATURES_SPEAKER_CSV", "./data/features_speaker.csv"))

# Perform clustering and save results
# perform_clustering_and_save(df, features, "B")
# perform_clustering_and_save(df, features1, "B1")
# perform_clustering_and_save(df, features2, "B2")
# perform_clustering_and_save(df, features3, "B3")
# perform_clustering_and_save(df, features4, "B4")
# perform_clustering_and_save(df, features5, "B5")

In [ ]:
# visual data
features6 = ["AU01_mean", "AU02_mean", "AU04_mean", "AU06_mean", "AU12_mean", 
                "AU15_mean", "AU20_mean", "AU25_mean",
                "pitch_count_percentage", "roll_count_percentage", "yaw_count_percentage",
                "valence_mean", "valence_std", "arousal_mean", "arousal_std"]

visual = pd.read_csv(os.environ.get("VISUAL_CSV", "./data/visual_241001-241104_N398.csv"))
perform_clustering_and_save(visual, features6, "visual")

In [ ]:
# combined data
features7 = ["AU01_mean", "AU02_mean", "AU04_mean", "AU06_mean", "AU12_mean", 
            "AU15_mean", "AU20_mean", "AU25_mean",
            "pitch_count_percentage", "roll_count_percentage", "yaw_count_percentage",
            "valence_mean", "valence_std", "arousal_mean", "arousal_std",
            "mean_f0", "std_f0", "VUV", "speaking_rate_w", "MFCC0",
            "F1", "F2",
            "falling_ratio", "rising_ratio", "rising-falling_ratio", "falling-rising_ratio",
            "arousal", "valence", "std_arousal", "std_valence"]
combined = pd.read_csv(os.environ.get("COMBINED_CSV", "./data/combined_241001-241104_N398_20250425.csv"))
perform_clustering_and_save(combined, features7, "combined")

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

df = pd.read_csv(os.environ.get("FEATURES_VIDEO_CSV", "./data/features_video.csv"))
scaler = StandardScaler()
X = scaler.fit_transform(df[features])
for k in range(2,11):
    
    kmeans = KMeans(n_clusters=k, random_state=42, n_init="auto")
    cluster_labels = kmeans.fit_predict(X)

    # Calculate metrics
    sil_score = silhouette_score(X, cluster_labels)
    db_score = davies_bouldin_score(X, cluster_labels)
    ch_score = calinski_harabasz_score(X, cluster_labels)

    if db_score <= 1:
        print(f'k = {k}')
        print(f'sil_score: {sil_score}')
        print(f'db_score: {db_score}')
        print(f'ch_score: {ch_score}')

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
root = os.environ.get("CLUSTER_EXPERIMENT_ROOT", "./data/experiment/N583/251126")
modality = "text_emo"
version = "33_k7_vauto"
# version = "42_k8_vauto"
# modality = "visual"
# version = "24_k6_v10"
# modality = "visual_audio"
# version = "42_k5_vauto"
# modality = "visual_text"
# version = "33_k6_v_auto"

# modality = "visual_emotion"
# version = "33_k7_v10"

# modality = "text"
# version = "24_k5_v_auto"

# modality = "audio_plus_emotion"
# version = "54_k4_v10"

# modality = "audio_emotion_only"
# version = "33_k7_v10"

df = pd.read_csv(f"{root}/{modality}/{modality}_cluster_{version}.csv")

# features
# visual data
features_v = ["AU01_mean", "AU02_mean", "AU04_mean", "AU06_mean", "AU12_mean", 
                "AU15_mean", "AU20_mean", "AU25_mean",
                "pitch_count_percentage", "roll_count_percentage", "yaw_count_percentage",
                "valence_mean", "valence_std", "arousal_mean", "arousal_std", "anger_percentage", "disgust_percentage",	
                "fear_percentage", "happiness_percentage", "sadness_percentage", "surprise_percentage","neutral_percentage"]
# audio data
features_a = ["mean_f0", "std_f0", "VUV", "speaking_rate_w", "MFCC0", "F1", "F2",
            "falling_ratio", "rising_ratio", "rising-falling_ratio", "falling-rising_ratio",
            "arousal", "valence", "std_arousal", "std_valence",
            "angry", "disgusted", "fearful", "happy", "neutral", "sad", "surprised"]

features_a_va = ["arousal", "valence", "std_arousal", "std_valence"]

features_a_emo = ["angry", "disgusted", "fearful", "happy", "neutral", "sad", "surprised"]

# features_t = ["WC", "Analytic", "Clout", "Authentic", "Tone", "WPS", "BigWords", "Dic",
#             "Linguistic", "function", "pronoun", "ppron", "i", "we", "you", "shehe", "they",
#             "ipron", "det", "article", "number", "prep", "auxverb", "adverb", "conj", "negate",
#             "verb", "adj", "quantity", "Drives", "affiliation", "achieve", "power", "Cognition",
#             "allnone", "cogproc", "insight", "cause", "discrep", "tentat", "certitude", "differ",
#             "memory", "Affect", "tone_pos", "tone_neg", "emotion", "emo_pos", "emo_neg", "emo_anx",
#             "emo_anger", "emo_sad", "swear", "Social", "socbehav", "prosocial", "polite", "conflict",
#             "moral", "comm", "socrefs", "family", "friend", "female", "male", "Culture", "politic",
#             "ethnicity", "tech", "Lifestyle", "leisure", "home", "work", "money", "relig", "Physical",
#             "health", "illness", "wellness", "mental", "substances", "sexual", "food", "death", "need",
#             "want", "acquire", "lack", "fulfill", "fatigue", "reward", "risk", "curiosity", "allure",
#             "Perception", "attention", "motion", "space", "visual", "auditory", "feeling", "time",
#             "focuspast", "focuspresent", "focusfuture", "Conversation", "netspeak", "assent", "nonflu",
#             "filler", "AllPunc", "Period", "Comma", "QMark", "Exclam", "Apostro", "OtherP"]

# features_t = [ "Analytic", "Clout", "Authentic", "Tone", "affiliation", "achieve", "power", "emo_pos", "emo_neg", "emo_anx",
#     "emo_anger", "emo_sad", "swear", "prosocial", "polite", "conflict",
#     "moral", "politic", "ethnicity", "relig", "risk", "motion"]

features_t = [ "Analytic", "Clout", "Authentic", "affiliation", "achieve", "power", "emo_pos", "emo_anx",
    "emo_anger", "emo_sad", "swear", "prosocial", "polite", "conflict", "we", "they", "moral", "risk", "motion",
    "anger_t", "disgust_t", "fear_t", "joy_t", "neutral_t", "sadness_t", "surprise_t"]

# Standardize only selected features
scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[features_t] = scaler.fit_transform(df[features_t])

# Group by cluster_a and calculate the mean
cluster_summary = df_scaled.groupby('kmeans_cluster')[features_t].mean().round(2)
# Count rows per cluster
cluster_counts = df_scaled['kmeans_cluster'].value_counts().sort_index()
# Add counts as a new column
cluster_summary['N'] = cluster_counts
cluster_summary.to_csv(f"{root}/{modality}_clusters_{version}_mean.csv")
